# Layer 1 Control Panel

Edit the **Control Parameters** cell, then run top to bottom.

## Setup
Run once. Usually no edits.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

# Find project root even if Jupyter starts inside notebooks/.
ROOT = Path.cwd().resolve()
if ROOT.name.lower() == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.layer1_universe.screen import run_layer1, preview_columns

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 160)

print(f"Project root: {ROOT}")

## Control Parameters
Main tweak cell.

In [ ]:
# DATA SOURCE
# live      = fresh Finviz scrape (DEFAULT - what you almost always want)
# latest-db = fast, uses last saved scrape (DEV ONLY - data will be stale)
# parquet   = load one parquet file
SOURCE = "live"

# FILE LOCATIONS
# Keep these rooted at ROOT so paths work from any notebook folder.
CONFIG_PATH = ROOT / "config" / "filters.yaml"
DB_PATH = ROOT / "data" / "fundamentals.db"
PARQUET_PATH = None  # example: ROOT / "data" / "universe" / "2026-05-17" / "universe.parquet"
OUTPUT_ROOT = ROOT / "data" / "universe"
OUTPUT_DATE = None  # None = today's folder, or use "2026-05-17"

# RUN BEHAVIOR
SAVE_OUTPUTS = True      # False = preview only
SAVE_LIVE_TO_DB = True   # only used for SOURCE = "live"
SHOW_ROWS = 20

# LONG SCREEN TWEAKS
# None = use config/filters.yaml
LONG_PE_MAX = None             # lower = stricter value filter
LONG_OPER_MARGIN_MIN = None    # higher = stricter quality filter
LONG_PROFIT_MARGIN_MIN = None  # higher = stricter profitability filter
LONG_52W_LOW_MAX = None        # lower = closer to 52-week low
LONG_52W_HIGH_MIN = None       # higher = farther below 52-week high

# SHORT SCREEN TWEAKS
# None = use config/filters.yaml
SHORT_PE_MIN = None            # higher = more expensive names only
SHORT_PERF_YEAR_MIN = None     # higher = stronger 1-year momentum only
SHORT_52W_HIGH_MAX = None      # lower = closer to 52-week high

OVERRIDES = {
    "finviz_long.pe_max": LONG_PE_MAX,
    "finviz_long.operating_margin_min": LONG_OPER_MARGIN_MIN,
    "finviz_long.net_profit_margin_min": LONG_PROFIT_MARGIN_MIN,
    "finviz_long.high_52w_distance_low_max": LONG_52W_LOW_MAX,
    "finviz_long.ath_distance_high_min": LONG_52W_HIGH_MIN,
    "finviz_short.pe_min": SHORT_PE_MIN,
    "finviz_short.performance_year_min": SHORT_PERF_YEAR_MIN,
    "finviz_short.high_52w_distance_high_max": SHORT_52W_HIGH_MAX,
}

# Shows only active overrides.
{k: v for k, v in OVERRIDES.items() if v is not None}

## Run

In [ ]:
result = run_layer1(
    source=SOURCE,
    config_path=CONFIG_PATH,
    db_path=DB_PATH,
    parquet_path=PARQUET_PATH,
    output_root=OUTPUT_ROOT,
    output_date=OUTPUT_DATE,
    save_outputs=SAVE_OUTPUTS,
    save_live_to_db=SAVE_LIVE_TO_DB,
    overrides=OVERRIDES,
    show_rows=SHOW_ROWS,
    print_summary=False,
)

long_candidates = result["long_candidates"]
short_candidates = result["short_candidates"]
waterfall = pd.DataFrame(result["diagnostics"]["filter_waterfall"])

summary = pd.DataFrame([
    ["source", result["source_label"]],
    ["output_folder", str(result["out_dir"])],
    ["universe_rows", len(result["universe"])],
    ["long_candidates", len(long_candidates)],
    ["short_candidates", len(short_candidates)],
    ["duplicate_tickers", result["manifest"].get("duplicate_tickers")],
], columns=["metric", "value"])

display(summary)

## Data Freshness + Schema Validation

This cell proves whether the run above used **live** Finviz data or a cached snapshot, and
validates the universe DataFrame against the new Pydantic `UniverseRow` schema
(Phase A foundation). Any rows that fail validation are surfaced here with their
rejection reason — this is the safety net before factor scoring.

In [ ]:
from datetime import datetime
from pydantic import ValidationError

from src.common.schemas import UniverseRow, normalize_finviz_columns, FilterConfig

# --- Freshness check ---
source_label = result["source_label"]
out_dir = result.get("out_dir")
is_live = source_label == "finviz-live" or "live" in source_label
freshness_icon = "[LIVE]" if is_live else "[CACHED - STALE!]"
print(f"{freshness_icon} Source: {source_label}")
print(f"Run output dir: {out_dir}")
print(f"Run time:       {datetime.now().isoformat(timespec='seconds')}")
if not is_live:
    print("WARNING: data is from a cached snapshot. Set SOURCE='live' in the Control")
    print("Parameters cell above and re-run to get current Finviz data.")

# --- Schema validation ---
universe_raw = result["universe"]
universe_canonical = normalize_finviz_columns(universe_raw)
run_id_for_validation = f"layer1_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

accepted, rejected = [], []
for _, row in universe_canonical.iterrows():
    payload = {"run_id": run_id_for_validation, **row.to_dict()}
    # Drop columns the schema does not know about (e.g. extra Finviz fields).
    keep = set(UniverseRow.model_fields.keys())
    payload = {k: v for k, v in payload.items() if k in keep}
    try:
        UniverseRow(**payload)
        accepted.append(row["ticker"])
    except ValidationError as err:
        first_err = err.errors()[0]
        rejected.append({
            "ticker": row.get("ticker", "?"),
            "field": ".".join(str(p) for p in first_err.get("loc", [])),
            "reason": first_err.get("msg", "unknown"),
        })

validation_summary = pd.DataFrame([
    ["universe rows (raw)", len(universe_raw)],
    ["accepted by UniverseRow", len(accepted)],
    ["rejected by UniverseRow", len(rejected)],
], columns=["metric", "value"])
display(validation_summary)

if rejected:
    print("\nTop rejection reasons:")
    rejected_df = pd.DataFrame(rejected)
    display(rejected_df["reason"].value_counts().head(10).rename_axis("reason").reset_index(name="count"))
    print("\nFirst 10 rejected tickers:")
    display(rejected_df.head(10))

print("\nCanonical column preview (first 3 rows):")
preview_cols = [c for c in ["ticker", "sector", "market_cap_usd", "pe_ratio", "operating_margin", "rsi_14"] if c in universe_canonical.columns]
display(universe_canonical[preview_cols].head(3))

## Filter Waterfall
Use this to see which filter is too strict or too loose.

In [ ]:
display(waterfall)

## Data Coverage
Low coverage means a field may be unreliable for filtering.

In [ ]:
coverage = pd.Series(result["manifest"]["critical_field_coverage"], name="coverage_pct")
coverage = (coverage * 100).round(2).rename_axis("field").reset_index()
display(coverage)

## Long Candidates

In [ ]:
# Sort tweak.
LONG_SORT_BY = ["52W High_num", "P/E_num"]
LONG_ASCENDING = [True, True]

if len(long_candidates):
    long_view = long_candidates.sort_values(LONG_SORT_BY, ascending=LONG_ASCENDING)
    display(long_view[preview_columns(long_view)].head(SHOW_ROWS))
else:
    print("No long candidates. Check the long waterfall.")

## Short Candidates

In [ ]:
# Sort tweak.
SHORT_SORT_BY = ["Perf Year_num", "P/E_num"]
SHORT_ASCENDING = [False, False]

if len(short_candidates):
    short_view = short_candidates.sort_values(SHORT_SORT_BY, ascending=SHORT_ASCENDING)
    display(short_view[preview_columns(short_view)].head(SHOW_ROWS))
else:
    print("No short candidates. Check the short waterfall.")

## Export Watchlists
Optional ticker-only files.

In [ ]:
EXPORT_WATCHLISTS = True

if EXPORT_WATCHLISTS and result["out_dir"] is not None:
    out_dir = Path(result["out_dir"])
    long_candidates[["Ticker"]].drop_duplicates().sort_values("Ticker").to_csv(out_dir / "watchlist_long.csv", index=False)
    short_candidates[["Ticker"]].drop_duplicates().sort_values("Ticker").to_csv(out_dir / "watchlist_short.csv", index=False)
    print(out_dir / "watchlist_long.csv")
    print(out_dir / "watchlist_short.csv")
else:
    print("Watchlist export skipped.")